Load cleaned data

In [15]:
import pandas as pd




df = pd.read_csv('data/processed/cleaned_reviews.csv')

# 🔥 IMPORTANT LINE
df = df.dropna(subset=['final_text', 'sentiment'])

X = df['final_text']
y = df['sentiment']


In [16]:
df = df.dropna(subset=['final_text', 'sentiment'])


Train–test split (VERY important)

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [18]:
print(len(X_train), len(X_test))


18112 4529


Feature Engineering → TF-IDF

In [19]:
print(X_train.isna().sum())
print(X_test.isna().sum())


0
0


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


print(X_train_tfidf.shape)


(18112, 5000)


In [21]:
print(X_train_tfidf.shape)


(18112, 5000)


exactly at the entry point of Stage 2.2.

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train_tfidf, y_train)



y_pred = lr_model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.823139765952749
              precision    recall  f1-score   support

    negative       0.61      0.45      0.52       474
     neutral       0.43      0.22      0.30       565
    positive       0.87      0.97      0.92      3490

    accuracy                           0.82      4529
   macro avg       0.64      0.55      0.58      4529
weighted avg       0.79      0.82      0.80      4529



Stage 2.3
Try Multiple Traditional ML Models

Model 1: Naive Bayes (fastest baseline)

In [23]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

y_pred_nb = nb_model.predict(X_test_tfidf)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))


Naive Bayes Accuracy: 0.7999558401413116
              precision    recall  f1-score   support

    negative       0.72      0.22      0.34       474
     neutral       0.46      0.09      0.16       565
    positive       0.81      0.99      0.89      3490

    accuracy                           0.80      4529
   macro avg       0.67      0.44      0.46      4529
weighted avg       0.76      0.80      0.74      4529



Model 2: Linear SVM (often strongest)

In [24]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(random_state=42)
svm_model.fit(X_train_tfidf, y_train)

y_pred_svm = svm_model.predict(X_test_tfidf)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))


SVM Accuracy: 0.8169573857363657
              precision    recall  f1-score   support

    negative       0.55      0.50      0.53       474
     neutral       0.40      0.25      0.31       565
    positive       0.89      0.95      0.92      3490

    accuracy                           0.82      4529
   macro avg       0.61      0.57      0.59      4529
weighted avg       0.79      0.82      0.80      4529



Model 3: Random Forest

In [25]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_tfidf, y_train)
y_pred_rf = rf_model.predict(X_test_tfidf)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))


Random Forest Accuracy: 0.7725767277544712


Model Comparison Table (VERY IMPORTANT)

In [27]:
import os

os.makedirs('results/metrics', exist_ok=True)


In [28]:
import pandas as pd
from sklearn.metrics import accuracy_score

results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Naive Bayes',
        'Linear SVM'
    ],
    'Accuracy': [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_svm)
    ]
})

results = results.sort_values('Accuracy', ascending=False)
print(results)

results.to_csv('results/metrics/model_comparison.csv', index=False)


                 Model  Accuracy
0  Logistic Regression  0.823140
2           Linear SVM  0.816957
1          Naive Bayes  0.799956
